# Okta Authorization Server Setup for AgentCore Gateway Demo

> **Security Notice — Trial Account Only**
>
> This notebook stores the `OKTA_API_TOKEN` locally in a `.env` file. This is acceptable for an **Okta trial/developer account** used for demo purposes, but is **not recommended for production Okta organizations**.
>
> - The Okta API token grants **full admin access** to the org — treat it like a root password.
> - The `.env` file is protected by `.gitignore`, but accidental commits or copy-paste into notebook output can still leak secrets.
> - For production use, store secrets in a secrets manager (e.g., AWS Secrets Manager, HashiCorp Vault) and use scoped OAuth 2.0 service apps instead of API tokens.
>
> **Do not run this notebook against a production Okta org.**

This notebook configures the **Okta authorization server**, creates the **SPA app for Claude Code**, and provisions **demo users**, required before running `01_agentcore_setup.ipynb`.

## Prerequisites (manual steps)

1. Create an Okta account at [https://developer.okta.com](https://developer.okta.com)
2. Create an **API Token** (Security > API > Tokens > Create Token)
3. Copy `.env.example` to `.env` and fill in:
   - `OKTA_DOMAIN` — e.g., `<your-okta-domain>.okta.com` (not the `-admin` domain)
   - `OKTA_API_TOKEN` — from step 2

## What this notebook configures

| Cell | What | Why |
|------|------|-----|
| 1 | Load `OKTA_DOMAIN` + `OKTA_API_TOKEN` | Only two env vars needed |
| 2 | Create custom scope + claims | `groups` scope, `groups` + `client_id` claims on auth server |
| 3 | Create access policy + rule | Allow grant types for all clients |
| 4 | Create SPA app for Claude Code | Public client (PKCE), writes `OKTA_SPA_CLIENT_ID` to `.env` |
| 5 | Configure default scopes | Required for Claude Code's MCP OAuth flow (RFC 9728) |
| 6 | Create demo users + groups | Alex (analysts), Jordan (finance-admins) for Cedar policies |

After this notebook completes:
- Run `01_agentcore_setup.ipynb` to deploy AWS infrastructure
- `05_strands_demo.ipynb` creates its own Web app when run

## Cell 1: Load Configuration

In [ ]:
%pip install -q -r requirements.txt

import os
import json
import urllib.parse
import base64
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

OKTA_DOMAIN = os.environ["OKTA_DOMAIN"]
OKTA_API_TOKEN = os.environ["OKTA_API_TOKEN"]
OKTA_ISSUER = f"https://{OKTA_DOMAIN}/oauth2/default"

HEADERS = {
    "Authorization": f"SSWS {OKTA_API_TOKEN}",
    "Content-Type": "application/json",
}

AUTH_SERVER_BASE = f"https://{OKTA_DOMAIN}/api/v1/authorizationServers/default"

# Verify API token works
resp = requests.get(f"https://{OKTA_DOMAIN}/api/v1/org", headers=HEADERS)
resp.raise_for_status()
org = resp.json()

print(f"Okta Domain:  {OKTA_DOMAIN}")
print(f"Okta Org:     {org.get('companyName', org.get('name', 'OK'))}")
print(f"Issuer:       {OKTA_ISSUER}")
print(f"\n✅ API token verified")

## Cell 2: Create Custom Scope and Claims

### First — scope vs. claim (they are not the same thing)

| | **Scope** | **Claim** |
|--|--|--|
| What it is | A **request for permission** — "this token may access X" | The **actual value** written into the JWT |
| Analogy | A checkbox on a consent screen ("☑ See your groups") | The data printed on the badge (`groups: ["analysts"]`) |
| Where it lives | On the authorization request | Inside the issued access token |

A **scope does not carry data** — it *unlocks* which claims are allowed to appear. A **claim only gets written into the token if its required scope was requested.** Think: scope = the *ask*, claim = the *answer*.

### What this cell creates on the `/oauth2/default` authorization server

- **`groups` scope** *(custom)* — declares the permission "this token may carry the user's group memberships." `consent: IMPLICIT` grants it automatically, with no consent screen.

- **`groups` claim** *(custom, `valueType: GROUPS`)* — injects the user's actual Okta groups (e.g. `analysts`, `finance-admins`) into the JWT, **but only when the `groups` scope is present** (`conditions.scopes: ["groups"]`). This is what Cedar uses downstream for group-based authorization.

- **`client_id` claim** *(custom, `valueType: EXPRESSION`)* — copies the calling app's client ID into a claim literally named `client_id`. Okta natively emits this as `cid`, but **AgentCore Gateway checks `allowedClients` against `client_id`**, so we remap it. Always included (no scope condition).

> **Gotchas**
> - Without the `client_id` claim, the Gateway returns `401 insufficient_scope`.
> - `group_filter_type: REGEX` is **required** whenever `valueType` is `GROUPS`. Omitting it fails with `Api validation failed: group_filter_type`.

> **Why it matters:** Standard OIDC scopes (`openid`, `profile`, `email`) don't expose groups or remap the client ID. These two pieces of identity — *which groups* the user is in and *which app* is calling — are exactly what the Gateway and Cedar need, so we define them as custom scope/claims here.

In [ ]:
# --- Create 'groups' scope ---
resp = requests.post(
    f"{AUTH_SERVER_BASE}/scopes",
    headers=HEADERS,
    json={
        "name": "groups",
        "description": "Access to user group memberships",
        "consent": "IMPLICIT",
    },
)

if resp.status_code in (200, 201):
    print(f"Created 'groups' scope (id: {resp.json()['id']})")
elif resp.status_code in (400, 409) and "unique" in resp.text.lower():
    print("'groups' scope already exists")
else:
    print(f"Error creating scope: {resp.status_code} — {resp.text[:300]}")

# --- Create custom claims ---
claims = [
    {
        "name": "groups",
        "status": "ACTIVE",
        "claimType": "RESOURCE",
        "valueType": "GROUPS",
        "value": ".*",
        "group_filter_type": "REGEX",
        "alwaysIncludeInToken": True,
        "conditions": {"scopes": ["groups"]},
    },
    {
        "name": "client_id",
        "status": "ACTIVE",
        "claimType": "RESOURCE",
        "valueType": "EXPRESSION",
        "value": "app.clientId",
        "alwaysIncludeInToken": True,
        "conditions": {"scopes": []},
    },
]

for claim in claims:
    resp = requests.post(
        f"{AUTH_SERVER_BASE}/claims",
        headers=HEADERS,
        json=claim,
    )
    if resp.status_code in (200, 201):
        print(f"Created '{claim['name']}' claim")
    elif resp.status_code in (400, 409) and "unique" in resp.text.lower():
        print(f"'{claim['name']}' claim already exists")
    else:
        print(f"Error creating '{claim['name']}': {resp.status_code} — {resp.text[:300]}")

print("\n✅ Custom scope and claims configured")

## Cell 3: Create Access Policy on Authorization Server

Scopes and claims define *what can go in a token*; an **access policy** is a different mechanism — it's the **gate to getting a token at all.** It answers four questions:

| Question | Controlled by | This demo |
|--|--|--|
| Is this **client** allowed to request a token? | `conditions.clients` | `ALL_CLIENTS` |
| Via which **grant types**? | `conditions.grantTypes` | `authorization_code`, `password`, `client_credentials`, `implicit` |
| Which **users**? | `conditions.people` | `groups: EVERYONE` |
| How long does the token **live**? | `actions.token` | 60-minute access token |

The grant types map to the flows used across the demos:

- `authorization_code` — Claude Code's SPA flow (PKCE)
- `password` — the Strands demo's direct username/password flow
- `client_credentials`, `implicit` — covered for completeness

> **`EVERYONE` is an admission rule, not a claim rule.** It means *"any user, regardless of group, may use this policy to obtain a token"* — it does **not** mean "put everyone's groups into the token." Whether the `groups` **claim** appears is decided separately by the claim's scope condition (Cell 2) plus the default scope (Cell 5):
>
> ```
> Cell 3 (this cell) → can this client/user get a token?     (the door)
> Cell 2 + Cell 5    → does the token carry the groups claim? (the badge)
> ```

> **Gotcha:** The `people` condition must use `"groups": { "include": ["EVERYONE"] }`, **not** `"everyone": ...`.

In [ ]:
POLICY_NAME = "Demo App Policy"

# Check if policy already exists
existing = requests.get(f"{AUTH_SERVER_BASE}/policies", headers=HEADERS)
existing_policy = next(
    (p for p in existing.json() if p["name"] == POLICY_NAME), None
)

if existing_policy:
    POLICY_ID = existing_policy["id"]
    print(f"Policy '{POLICY_NAME}' already exists (id: {POLICY_ID})")
else:
    resp = requests.post(
        f"{AUTH_SERVER_BASE}/policies",
        headers=HEADERS,
        json={
            "name": POLICY_NAME,
            "description": "Allow password grant for demo app",
            "type": "OAUTH_AUTHORIZATION_POLICY",
            "status": "ACTIVE",
            "priority": 1,
            "conditions": {
                "clients": {"include": ["ALL_CLIENTS"]}
            },
        },
    )
    resp.raise_for_status()
    POLICY_ID = resp.json()["id"]
    print(f"Created policy '{POLICY_NAME}' (id: {POLICY_ID})")

# Create rule on the policy
RULE_NAME = "Allow all grant types"
existing_rules = requests.get(
    f"{AUTH_SERVER_BASE}/policies/{POLICY_ID}/rules", headers=HEADERS
)
existing_rule = next(
    (r for r in existing_rules.json() if r["name"] == RULE_NAME), None
)

if existing_rule:
    print(f"Rule '{RULE_NAME}' already exists")
else:
    resp = requests.post(
        f"{AUTH_SERVER_BASE}/policies/{POLICY_ID}/rules",
        headers=HEADERS,
        json={
            "name": RULE_NAME,
            "type": "RESOURCE_ACCESS",
            "status": "ACTIVE",
            "priority": 1,
            "conditions": {
                "grantTypes": {
                    "include": ["implicit", "password", "client_credentials", "authorization_code"]
                },
                "people": {
                    "groups": {"include": ["EVERYONE"]}
                },
                "scopes": {"include": ["*"]},
            },
            "actions": {
                "token": {
                    "accessTokenLifetimeMinutes": 60,
                    "refreshTokenLifetimeMinutes": 0,
                    "refreshTokenWindowMinutes": 10080,
                }
            },
        },
    )
    resp.raise_for_status()
    print(f"Created rule '{RULE_NAME}'")

print("\n\u2705 Access policy configured")

## Cell 4: Create SPA Application for Claude Code

An app registration in Okta represents the **client** that requests tokens. Claude Code is a desktop CLI with no secure backend to hold a secret, so it uses **Authorization Code + PKCE** — a *public client* flow that proves possession via a one-time code challenge instead of a stored secret. In Okta this maps to a **SPA (Single Page Application)** type app.

| | SPA App (Claude Code) |
|--|---|
| **Application type** | `browser` (public client) |
| **Auth method** | `none` (PKCE only, no client secret) |
| **Grant type** | `authorization_code` |
| **Redirect URI** | `http://localhost:8400/callback` |
| **Token flow** | Browser popup |

The app is assigned to the **Everyone** group so any demo user can sign in. Its **client ID** is what the `client_id` claim (Cell 2) carries and what the Gateway matches against `allowedClients`. We write it to `.env` as `OKTA_SPA_CLIENT_ID` for use by later notebooks.

> **Note:** Demo *users* are not assigned to this app here — group/user-to-app assignment happens in each demo notebook (e.g. `05_strands_demo.ipynb`), which creates its own app.

In [ ]:
import re

CALLBACK_PORT = 8400  # Claude Code's OAuth callback port
SPA_LABEL = "AgentCore Gateway - Claude Code (SPA)"


def set_env_var(content, key, value):
    """Set or update a key=value pair in .env file content."""
    pattern = re.compile(rf"^{re.escape(key)}=.*$", re.MULTILINE)
    if pattern.search(content):
        return pattern.sub(f"{key}={value}", content)
    else:
        return content.rstrip() + f"\n{key}={value}\n"


# --- Check if SPA app already exists ---
existing_apps = requests.get(
    f"https://{OKTA_DOMAIN}/api/v1/apps?q={requests.utils.quote(SPA_LABEL)}&limit=10",
    headers=HEADERS,
)
existing_app = next(
    (a for a in existing_apps.json() if isinstance(a, dict) and a.get("label") == SPA_LABEL), None
)

if existing_app:
    spa_app = existing_app
    SPA_CLIENT_ID = spa_app["credentials"]["oauthClient"]["client_id"]
    print(f"SPA app '{SPA_LABEL}' already exists")
    print(f"  Client ID: {SPA_CLIENT_ID}")
else:
    resp = requests.post(
        f"https://{OKTA_DOMAIN}/api/v1/apps",
        headers=HEADERS,
        json={
            "name": "oidc_client",
            "label": SPA_LABEL,
            "signOnMode": "OPENID_CONNECT",
            "settings": {
                "oauthClient": {
                    "application_type": "browser",
                    "grant_types": ["authorization_code"],
                    "response_types": ["code"],
                    "redirect_uris": [f"http://localhost:{CALLBACK_PORT}/callback"],
                    "post_logout_redirect_uris": [f"http://localhost:{CALLBACK_PORT}"],
                }
            },
            "credentials": {
                "oauthClient": {
                    "token_endpoint_auth_method": "none",
                }
            },
        },
    )
    resp.raise_for_status()
    spa_app = resp.json()
    SPA_CLIENT_ID = spa_app["credentials"]["oauthClient"]["client_id"]
    print(f"Created SPA app: {spa_app['label']}")
    print(f"  Client ID: {SPA_CLIENT_ID}")
    print(f"  Auth method: {spa_app['credentials']['oauthClient']['token_endpoint_auth_method']}")
    print(f"  Redirect URI: http://localhost:{CALLBACK_PORT}/callback")

# --- Assign app to Everyone group ---
everyone_resp = requests.get(
    f"https://{OKTA_DOMAIN}/api/v1/groups?q=Everyone&limit=1",
    headers=HEADERS,
)
everyone_group_id = everyone_resp.json()[0]["id"]

assign_resp = requests.put(
    f"https://{OKTA_DOMAIN}/api/v1/apps/{spa_app['id']}/groups/{everyone_group_id}",
    headers=HEADERS,
)
if assign_resp.status_code in (200, 204):
    print(f"\nAssigned to Everyone group")
else:
    print(f"\nWarning: Could not assign to Everyone group: {assign_resp.status_code}")

# --- Write SPA client ID to .env ---
env_path = os.path.join(os.getcwd(), ".env")
if os.path.exists(env_path):
    with open(env_path) as f:
        env_content = f.read()
else:
    env_content = ""

env_content = set_env_var(env_content, "OKTA_SPA_CLIENT_ID", SPA_CLIENT_ID)

with open(env_path, "w") as f:
    f.write(env_content)

print(f"\nSPA Client ID: {SPA_CLIENT_ID}")
print(f"Written to .env as OKTA_SPA_CLIENT_ID")

## Cell 5: Configure Default Scopes

### What is a "default scope"?

A **default scope** is the permission the authorization server falls back to **when the client requests no scope at all**. It is the OAuth equivalent of a fallback policy that applies when a request arrives without specifying any permissions.

### Why this is required here

Claude Code follows the MCP OAuth spec (**RFC 9728**) and sends a `resource` parameter in the authorization request — but it does **not** send a `scope` parameter. From Okta's point of view, that's a request asking for *nothing*.

Okta's rule: if no `scope` is requested, fall back to whatever is marked **default** on the authorization server. If nothing is marked default, it errors out:

> *"The authorization server resource does not have any configured default scopes, 'scope' must be provided."*

By marking the custom **`groups` scope as default**, a scope-less request from Claude Code still resolves to the `groups` scope — which (from Cell 2) is the condition that unlocks the **`groups` claim** in the token.

```
no scope requested  →  default scope (groups) applies  →  groups claim included  →  Cedar can authorize by group
```

> **Note:** System scopes (`openid`, `profile`, …) cannot be set as default via the API, but custom scopes like `groups` can. We also publish `groups` (`metadataPublish: ALL_CLIENTS`) so it appears in the OpenID discovery document.

In [ ]:
# --- Set custom scopes as default ---
# System scopes (openid, profile, etc.) cannot be set as default via API,
# but custom scopes like "groups" can. We also publish "groups" so it appears
# in the OpenID discovery document.

scopes_resp = requests.get(
    f"https://{OKTA_DOMAIN}/api/v1/authorizationServers/default/scopes",
    headers=HEADERS,
)
scopes = scopes_resp.json()

for scope in scopes:
    if scope["name"] == "groups" and not scope.get("default"):
        # Set groups scope as default
        scope_update = {k: v for k, v in scope.items() if k != "_links"}
        scope_update["default"] = True
        scope_update["metadataPublish"] = "ALL_CLIENTS"

        update_resp = requests.put(
            f"https://{OKTA_DOMAIN}/api/v1/authorizationServers/default/scopes/{scope['id']}",
            headers=HEADERS,
            json=scope_update,
        )
        if update_resp.status_code == 200:
            print(f"Set '{scope['name']}' as default scope")
        else:
            print(f"Warning: Could not update '{scope['name']}': {update_resp.text[:200]}")
    elif scope["name"] == "groups" and scope.get("default"):
        print(f"'{scope['name']}' is already a default scope")

# Show final scope state
print("\nAuthorization server scopes:")
for scope in scopes:
    default = "DEFAULT" if scope.get("default") else ""
    system = "system" if scope.get("system") else "custom"
    print(f"  {scope['name']:20s} {system:8s} {default}")

## Cell 6: Create Demo Users and Groups

Creates the insurance demo personas in Okta. These users are needed for Cedar policy principal matching in `01_agentcore_setup.ipynb`.

| User | Group | Role | Cedar Access |
|------|-------|------|-------------|
| Alex | `analysts` | Customer Service Rep | PolicyLookup only |
| Jordan | `finance-admins` | Claims Adjuster | PolicyLookup + ClaimsData |

> **Note:** Users are NOT assigned to any OIDC app here — that happens in each demo notebook (e.g., `05_strands_demo.ipynb`) after it creates its own OIDC app.

In [ ]:
# --- Configuration ---
ALEX_USERNAME = os.environ.get("ALEX_USERNAME") or f"alex@{OKTA_DOMAIN}"
# Passwords come from .env if previously set; otherwise generate a strong
# random one and persist it to .env. This avoids hard-coding demo creds.
import secrets
import string
def _gen_password(length: int = 20) -> str:
    alphabet = string.ascii_letters + string.digits + "!@#$%^&*"
    # ensure at least one of each required character class for Okta password policy
    while True:
        pw = "".join(secrets.choice(alphabet) for _ in range(length))
        if (any(c.islower() for c in pw) and any(c.isupper() for c in pw)
                and any(c.isdigit() for c in pw) and any(c in "!@#$%^&*" for c in pw)):
            return pw

ALEX_PASSWORD = os.environ.get("ALEX_PASSWORD") or _gen_password()
JORDAN_USERNAME = os.environ.get("JORDAN_USERNAME") or f"jordan@{OKTA_DOMAIN}"
JORDAN_PASSWORD = os.environ.get("JORDAN_PASSWORD") or _gen_password()

GROUPS = {
    "analysts": {"description": "Customer service representatives"},
    "finance-admins": {"description": "Claims adjusters and finance administrators"},
}

USERS = {
    "Alex": {
        "firstName": "Alex",
        "lastName": "Demo",
        "login": ALEX_USERNAME,
        "email": ALEX_USERNAME,
        "password": ALEX_PASSWORD,
        "group": "analysts",
    },
    "Jordan": {
        "firstName": "Jordan",
        "lastName": "Demo",
        "login": JORDAN_USERNAME,
        "email": JORDAN_USERNAME,
        "password": JORDAN_PASSWORD,
        "group": "finance-admins",
    },
}

# --- Create groups ---
group_ids = {}
for name, meta in GROUPS.items():
    resp = requests.post(
        f"https://{OKTA_DOMAIN}/api/v1/groups",
        headers=HEADERS,
        json={"profile": {"name": name, "description": meta["description"]}},
    )
    if resp.status_code == 200:
        group_ids[name] = resp.json()["id"]
        print(f"Created group '{name}' (id: {group_ids[name]})")
    else:
        search = requests.get(
            f"https://{OKTA_DOMAIN}/api/v1/groups?q={name}&limit=1",
            headers=HEADERS,
        )
        match = next((g for g in search.json() if g["profile"]["name"] == name), None)
        if match:
            group_ids[name] = match["id"]
            print(f"Group '{name}' already exists (id: {group_ids[name]})")
        else:
            print(f"Error creating group '{name}': {resp.status_code} — {resp.text[:200]}")

# --- Create users ---
user_ids = {}
for display_name, info in USERS.items():
    resp = requests.post(
        f"https://{OKTA_DOMAIN}/api/v1/users?activate=true",
        headers=HEADERS,
        json={
            "profile": {
                "firstName": info["firstName"],
                "lastName": info["lastName"],
                "email": info["email"],
                "login": info["login"],
            },
            "credentials": {
                "password": {"value": info["password"]}
            },
        },
    )
    if resp.status_code == 200:
        user_ids[display_name] = resp.json()["id"]
        print(f"\nCreated user '{display_name}' ({info['login']})")
    else:
        err = resp.json()
        if "already exists" in str(err).lower() or resp.status_code == 400:
            search = requests.get(
                f"https://{OKTA_DOMAIN}/api/v1/users?q={info['login']}&limit=1",
                headers=HEADERS,
            )
            users = search.json()
            match = next((u for u in users if u["profile"]["login"] == info["login"]), None)
            if match:
                user_ids[display_name] = match["id"]
                print(f"\nUser '{display_name}' already exists ({info['login']})")
            else:
                print(f"\nError: could not find or create '{display_name}': {err}")
                continue
        else:
            print(f"\nError creating '{display_name}': {resp.status_code} — {resp.text[:300]}")
            continue

    # Assign user to group
    group_name = info["group"]
    if group_name in group_ids and display_name in user_ids:
        resp = requests.put(
            f"https://{OKTA_DOMAIN}/api/v1/groups/{group_ids[group_name]}/users/{user_ids[display_name]}",
            headers=HEADERS,
        )
        if resp.status_code in (200, 204):
            print(f"  Assigned to group '{group_name}'")
        else:
            print(f"  Warning: assign to group returned {resp.status_code}")

# Write user credentials to .env
with open(env_path) as f:
    env_content = f.read()

env_content = set_env_var(env_content, "ALEX_USERNAME", ALEX_USERNAME)
env_content = set_env_var(env_content, "ALEX_PASSWORD", ALEX_PASSWORD)
env_content = set_env_var(env_content, "JORDAN_USERNAME", JORDAN_USERNAME)
env_content = set_env_var(env_content, "JORDAN_PASSWORD", JORDAN_PASSWORD)

with open(env_path, "w") as f:
    f.write(env_content)

print(f"\n--- Demo Users ---")
print(f"Alex: {ALEX_USERNAME} / group: analysts")
print(f"Jordan:   {JORDAN_USERNAME} / group: finance-admins")
print(f"\n✅ Users, groups, and credentials written to .env")